# Stage 2 — 재랭커 학습·평가 (`stage2.ipynb`)

Stage1이 뽑은 후보의 **순위를 다시 매기는** cross-encoder 재랭커를 골드셋 라벨로 파인튜닝하고,
임베딩 베이스라인과 **Recall@10 / MRR**로 비교한다.

실행 순서: 위에서부터 셀을 차례로 실행. **런타임 → GPU** 권장.

| 셀 | 내용 |
|---|---|
| 1 | 설치 + `reranker.py` 생성 |
| 2 | 라벨링 CSV 로드 → 학습 예시 구성 |
| 3 | time-split (과거 학습 / 최근 평가) |
| 4 | 재랭커 학습 |
| 5 | 평가 (재랭커 vs 임베딩 베이스라인) |
| 6 | 재랭킹 데모 + 모델 저장 |


## 1. 설치 + `reranker.py` 생성
CrossEncoder 학습 API 안정성을 위해 `sentence-transformers`를 고정 버전으로 설치한다.

In [ ]:
%pip install -q "sentence-transformers==2.7.0"

In [ ]:
%%writefile reranker.py
"""
Stage 2 재랭커 (cross-encoder) — Stage1 후보의 순위를 다시 매긴다.

설계
- 입력: (프로필 텍스트, 논문 제목+초록) 쌍 → 관련도 점수 1개
- 모델: sentence-transformers CrossEncoder 파인튜닝
- 학습 라벨: 골드셋 0/1 (label=1 positive, label=0 및 hard negative)
- 평가: 프로필별 time-split(과거 학습 / 최근 평가), Recall@k / MRR
- 베이스라인: 임베딩(cosine) 순위와 비교해 "재랭커가 이기는지" 확인

torch / sentence-transformers가 필요하므로 GPU(Colab)에서 실행 권장.
이 모듈은 db/chroma에 의존하지 않는다 — 라벨·메타·프로필 텍스트를 인자로 받는다.
서버에서 서빙하려면 load_reranker()로 저장 모델을 불러 rerank()만 쓰면 된다.
"""

from __future__ import annotations

import random
from collections import defaultdict
from typing import List, Optional

# 사전학습 랭킹 cross-encoder에서 출발해 소량 골드셋으로 파인튜닝(데이터가 적어 from-scratch 지양)
DEFAULT_BASE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
ABSTRACT_CHARS = 1200  # 초록을 잘라 입력 길이 관리


def doc_text(meta: dict) -> str:
    """논문 메타(title, abstract_clean) → 재랭커 입력 문서 텍스트."""
    title = (meta.get("title") or "").strip()
    abstract = (meta.get("abstract_clean") or meta.get("abstract") or "").strip()[:ABSTRACT_CHARS]
    return f"{title}. {abstract}".strip(". ").strip()


def build_examples(labels, meta_by_id, profile_text_by_id, prefer_human=True) -> List[dict]:
    """골드셋 라벨 → 학습/평가 예시 리스트.

    labels: [{profile_id, arxiv_id, label, source, tag, labeler, ...}]
    meta_by_id: {arxiv_id: {title, abstract_clean, submitted_date, ...}}
    profile_text_by_id: {profile_id: 영어 프로필 텍스트}
    같은 (profile, arxiv)에 여러 labeler가 있으면 사람 라벨 우선(prefer_human).
    반환 항목: {profile_id, arxiv_id, query, doc, label, submitted_date, source, tag}
    """
    chosen = {}
    for lab in labels:
        key = (lab["profile_id"], lab["arxiv_id"])
        prev = chosen.get(key)
        if prev is None:
            chosen[key] = lab
        elif prefer_human:
            is_human = (lab.get("labeler") or "").lower() != "judge"
            prev_judge = (prev.get("labeler") or "").lower() == "judge"
            if is_human and prev_judge:
                chosen[key] = lab

    out = []
    for (pid, aid), lab in chosen.items():
        query = profile_text_by_id.get(pid)
        meta = meta_by_id.get(aid)
        if not query or not meta:
            continue
        d = doc_text(meta)
        if not d:
            continue
        try:
            label = int(lab["label"])
        except (TypeError, ValueError):
            continue  # 빈 칸 / uncertain 등은 제외
        out.append({
            "profile_id": pid, "arxiv_id": aid, "query": query, "doc": d,
            "label": label, "submitted_date": meta.get("submitted_date", ""),
            "source": lab.get("source"), "tag": lab.get("tag"),
        })
    return out


def time_split(examples: List[dict], test_frac: float = 0.3):
    """프로필별로 submitted_date 오름차순 정렬 후 뒤쪽(최근) test_frac을 test로 뗀다.
    과거로 학습하고 최근으로 평가해 시간 누수를 막는다. 반환: (train, test)."""
    by_p = defaultdict(list)
    for e in examples:
        by_p[e["profile_id"]].append(e)
    train, test = [], []
    for _pid, lst in by_p.items():
        lst = sorted(lst, key=lambda e: e["submitted_date"] or "")
        n_test = max(1, round(len(lst) * test_frac)) if len(lst) >= 4 else 0
        cut = len(lst) - n_test
        train += lst[:cut]
        test += lst[cut:]
    return train, test


def train_reranker(train_examples: List[dict], base_model: str = DEFAULT_BASE_MODEL,
                   epochs: int = 3, batch_size: int = 16, lr: float = 2e-5, seed: int = 42):
    """CrossEncoder 파인튜닝. 반환: 학습된 CrossEncoder."""
    import torch
    from torch.utils.data import DataLoader
    from sentence_transformers import CrossEncoder, InputExample

    random.seed(seed)
    torch.manual_seed(seed)

    samples = [InputExample(texts=[e["query"], e["doc"]], label=float(e["label"]))
               for e in train_examples]
    loader = DataLoader(samples, shuffle=True, batch_size=batch_size)
    model = CrossEncoder(base_model, num_labels=1, max_length=512)
    warmup = max(1, int(0.1 * len(loader) * epochs))
    model.fit(train_dataloader=loader, epochs=epochs, warmup_steps=warmup,
              optimizer_params={"lr": lr}, show_progress_bar=True)
    return model


def _predict(model, query: str, docs: List[str], batch_size: int = 32):
    if not docs:
        return []
    return list(model.predict([[query, d] for d in docs],
                              batch_size=batch_size, show_progress_bar=False))


def rerank(model, query: str, candidates: List[dict], batch_size: int = 32) -> List[dict]:
    """candidates: [{arxiv_id, doc, ...}] → 각 항목에 rerank_score를 넣고 내림차순 정렬해 반환."""
    scores = _predict(model, query, [c["doc"] for c in candidates], batch_size)
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)
    return sorted(candidates, key=lambda c: c["rerank_score"], reverse=True)


# ── 지표 ──────────────────────────────────────────────
def recall_at_k(labels_ranked: List[int], k: int) -> Optional[float]:
    rel = sum(labels_ranked)
    if rel == 0:
        return None  # positive가 없는 프로필은 평균에서 제외
    return sum(labels_ranked[:k]) / rel


def mrr(labels_ranked: List[int]) -> float:
    for i, label in enumerate(labels_ranked, 1):
        if label == 1:
            return 1.0 / i
    return 0.0


def _macro(values) -> float:
    vals = [v for v in values if v is not None]
    return sum(vals) / len(vals) if vals else 0.0


def _embed_order(embed_model, query: str, docs: List[str]) -> List[int]:
    """임베딩(cosine) 기준 내림차순 인덱스 — Stage1 임베딩 순위와 동일한 베이스라인."""
    import numpy as np
    q = embed_model.encode([query], normalize_embeddings=True)[0]
    d = embed_model.encode(docs, normalize_embeddings=True)
    sims = d @ q
    return list(np.argsort(-sims))


def evaluate(model, test_examples: List[dict], embed_model=None, k: int = 10) -> dict:
    """프로필별로 재랭커(그리고 embed_model을 주면 임베딩 베이스라인) Recall@k / MRR을 재고
    macro 평균한다. 반환: {reranker, baseline|None, per_profile, k}."""
    by_p = defaultdict(list)
    for e in test_examples:
        by_p[e["profile_id"]].append(e)

    rr_recall, rr_mrr, bl_recall, bl_mrr, per = [], [], [], [], {}
    for pid, lst in by_p.items():
        labels = [e["label"] for e in lst]
        docs = [e["doc"] for e in lst]
        query = lst[0]["query"]

        scores = _predict(model, query, docs)
        order = sorted(range(len(lst)), key=lambda i: scores[i], reverse=True)
        rr_sorted = [labels[i] for i in order]
        r_recall, r_mrr = recall_at_k(rr_sorted, k), mrr(rr_sorted)
        rr_recall.append(r_recall)
        rr_mrr.append(r_mrr)
        entry = {"n": len(lst), "pos": sum(labels),
                 f"reranker_recall@{k}": r_recall, "reranker_mrr": r_mrr}

        if embed_model is not None:
            b_order = _embed_order(embed_model, query, docs)
            bl_sorted = [labels[i] for i in b_order]
            b_recall, b_mrr = recall_at_k(bl_sorted, k), mrr(bl_sorted)
            bl_recall.append(b_recall)
            bl_mrr.append(b_mrr)
            entry[f"baseline_recall@{k}"] = b_recall
            entry["baseline_mrr"] = b_mrr
        per[pid] = entry

    result = {
        "reranker": {f"recall@{k}": _macro(rr_recall), "mrr": _macro(rr_mrr)},
        "baseline": None, "per_profile": per, "k": k,
    }
    if embed_model is not None:
        result["baseline"] = {f"recall@{k}": _macro(bl_recall), "mrr": _macro(bl_mrr)}
    return result


def save_reranker(model, path: str) -> None:
    model.save(path)


def load_reranker(path: str):
    from sentence_transformers import CrossEncoder
    return CrossEncoder(path)


## 2. 라벨링 CSV 로드 → 학습 예시
사람이 라벨링을 끝낸 골드셋 CSV(`P1.csv` … `P12.csv`)를 읽는다.
`label` 칸이 `0` 또는 `1`인 행만 사용하고, 빈 칸·`uncertain`은 제외한다.

- **Drive에 있으면** `LABEL_DIR`을 그 폴더로 지정
- **없으면** 실행 시 파일 업로드 창이 뜬다 (여러 CSV 선택)

In [ ]:
import glob, os
import pandas as pd
from reranker import doc_text

# profile_id -> 영어 프로필 텍스트 (빌드 시점 profiles.json에서 주입)
PROFILE_TEXT = {
"P1": "I am interested in grasping within robot manipulation. I especially care about stable grasping using tactile sensors and force control, and about implementing human-hand grasping mechanisms on robots. I prefer optimization-based methods (contact mechanics, force closure, optimization approaches to grasp planning) over learning-based approaches.",
"P2": "I am interested in Vision-Language-Action (VLA) models, especially how VLA connects to manipulation tasks and how actions are represented and processed. Rather than the VLA architecture or training method itself, I care more about which real manipulation tasks VLA can be applied to.",
"P3": "I am interested in off-road navigation. Compared to indoor navigation, I am curious how robots perceive and reason about unstructured outdoor environments, and whether the perception-processing pipeline itself differs between indoor and outdoor settings.",
"P4": "I am interested in how robots perceive 3D objects and how that perception is converted into representations for manipulation. In particular, how perception data such as point clouds and 6D pose is passed to downstream action planning, and the representation and processing pipeline involved.",
"P5": "I research computer vision for autonomous driving. I am interested in 3D object detection, BEV (Bird's Eye View), and multi-camera perception. I want to exclude LiDAR-only research and path/trajectory planning research.",
"P6": "I research medical image analysis, focused on applying foundation models (SAM, MedSAM, DINO, MAE) to medical image segmentation and detection. I prefer research using prompt learning, adapters, parameter-efficient fine-tuning (PEFT), and vision-language models for medical imaging. I want general natural-image foundation model research ranked lower.",
"P7": "I research image generation. I am more interested in diffusion models than GANs, and want image editing and inpainting papers prioritized. I exclude research that deals only with text generation.",
"P8": "I research computer vision, especially test-time adaptation (TTA) and domain generalization. I want methods that adapt a model at test time without additional training data, and online adaptation research, prioritized.",
"P9": "I am interested in LLM inference efficiency, especially techniques that serve an already-trained model cheaper and faster, such as speculative decoding, KV cache compression, and post-training quantization. Rather than pretraining scaling laws or training-time efficiency, I focus on inference-time optimization of deployed models.",
"P10": "I am interested in retrieval-augmented generation, especially how retrieval quality affects the factuality of generated answers, and citation grounding (identifying which source documents an answer came from). For hallucination detection and mitigation I do not mind whether the approach uses internal model signals or external verification. Pure IR ranking research without any generation component is outside my interest.",
"P11": "I am interested in LLM agents that use external tools: how reliably function calling and tool use work in real environments, and how role division and multi-step planning happen when multiple agents collaborate. I focus on agents in software/API environments rather than game-playing RL agents or embodied robotic agents.",
"P12": "I am interested in how to evaluate LLM systems: how far LLM-as-a-judge can replace human evaluation, judge biases (position bias, self-preference) and reliability, and benchmark design and contamination detection. I prefer research that tackles the evaluation problem itself over papers that only report benchmark scores without a new evaluation methodology."
}

EMBED_MODEL = "BAAI/bge-small-en-v1.5"   # Stage1과 동일 임베딩(베이스라인용)
LABEL_DIR = "labeling"                   # Drive 경로로 바꿔도 됨

paths = sorted(glob.glob(f"{LABEL_DIR}/*.csv"))
if not paths:
    from google.colab import files
    up = files.upload()          # P1.csv ... P12.csv 업로드
    paths = sorted(up.keys())

def load_examples(paths):
    exs, skipped = [], 0
    for path in paths:
        pid = os.path.splitext(os.path.basename(path))[0]
        q = PROFILE_TEXT.get(pid)
        if not q:
            print("프로필 텍스트 없음 → 스킵:", pid); continue
        df = pd.read_csv(path)
        for _, r in df.iterrows():
            lab = str(r.get("label", "")).strip()
            if lab not in ("0", "1"):
                skipped += 1; continue
            d = doc_text({"title": r.get("title", ""), "abstract_clean": r.get("abstract_clean", "")})
            if not d:
                continue
            exs.append({"profile_id": pid, "arxiv_id": str(r.get("arxiv_id", "")),
                        "query": q, "doc": d, "label": int(lab),
                        "submitted_date": str(r.get("submitted_date", "")),
                        "source": r.get("source"), "tag": r.get("tag")})
    return exs, skipped

examples, skipped = load_examples(paths)
from collections import Counter
pos = sum(e["label"] for e in examples)
print(f"예시 {len(examples)}개 (positive {pos} / negative {len(examples)-pos}), 제외(빈칸·uncertain) {skipped}개")
print("프로필별 예시 수:", dict(Counter(e["profile_id"] for e in examples)))

## 3. time-split
프로필별로 제출일 기준 최근 30%를 평가셋으로 뗀다 (과거→학습, 최근→평가로 시간 누수 방지).

In [ ]:
from reranker import time_split
train, test = time_split(examples, test_frac=0.3)
print(f"train {len(train)}개 / test {len(test)}개")

## 4. 재랭커 학습
사전학습 랭킹 cross-encoder(`ms-marco-MiniLM-L-6-v2`)에서 출발해 골드셋으로 파인튜닝한다.
데이터가 적으므로 from-scratch가 아니라 파인튜닝이다. (GPU면 수 분)

In [ ]:
from reranker import train_reranker
model = train_reranker(train, epochs=3, batch_size=16)
print("학습 완료")

## 5. 평가 — 재랭커 vs 임베딩 베이스라인
같은 평가셋을 두 방식으로 정렬해 Recall@10 / MRR를 비교한다.
베이스라인은 Stage1과 동일한 임베딩(cosine) 순위다.

In [ ]:
from sentence_transformers import SentenceTransformer
from reranker import evaluate

embed = SentenceTransformer(EMBED_MODEL)
res = evaluate(model, test, embed_model=embed, k=10)

b, r = res["baseline"], res["reranker"]
print(f"{'지표':<12}{'베이스라인(임베딩)':>18}{'재랭커':>12}")
print("-" * 44)
print(f"{'Recall@10':<12}{b['recall@10']:>18.4f}{r['recall@10']:>12.4f}")
print(f"{'MRR':<12}{b['mrr']:>18.4f}{r['mrr']:>12.4f}")
print(f"\n개선: Recall@10 {r['recall@10']-b['recall@10']:+.4f}, MRR {r['mrr']-b['mrr']:+.4f}")

In [ ]:
# 프로필별 상세
import pandas as pd
pd.DataFrame(res["per_profile"]).T

## 6. 재랭킹 데모 + 모델 저장
한 프로필의 평가 후보를 재랭킹해 상위권에 정답(label=1)이 올라오는지 눈으로 확인하고, 모델을 저장한다.

In [ ]:
from reranker import rerank

pid = test[0]["profile_id"]
cands = [dict(e) for e in test if e["profile_id"] == pid]
ranked = rerank(model, PROFILE_TEXT[pid], cands)
print(f"[{pid}] 재랭킹 상위 8 (★=정답)")
for c in ranked[:8]:
    star = "★" if c["label"] == 1 else " "
    print(f"  {star} {c['rerank_score']:+.3f}  {c['arxiv_id']:<16} {c['doc'][:70]}")

In [ ]:
from reranker import save_reranker
save_reranker(model, "stage2_reranker")
print("저장: ./stage2_reranker  (Drive 보관 시: !cp -r stage2_reranker /content/drive/MyDrive/<경로>)")